# Load Dataset

In [4]:
################################################################################
# Load WUSTL-IIoT population CSV and split into per-class train/test sets
# (multi-class evaluation — Normal, DoS, Reconn only)
################################################################################

import pandas as pd
import numpy as np
import os
from tabulate import tabulate

dataset_name     = "wustl-iiot"
LABEL_COL        = "Traffic"   # multi-class label column in WUSTL-IIoT
NORMAL_CLASS     = "normal"
KEEP_CLASSES     = ["normal", "DoS", "Reconn"]  # Backdoor/CommInj excluded
TARGET_PER_CLASS = 8_000       # Reconn ceiling is 8,240; 8k is a clean, balanced cap

df_raw = pd.read_csv(os.getcwd() + '/data/population.csv',  low_memory=False)
print("Shape:", df_raw.shape)

# Filter to the three target classes
assert LABEL_COL in df_raw.columns, f"Column '{LABEL_COL}' not found."
df_raw = df_raw[df_raw[LABEL_COL].isin(KEEP_CLASSES)]
print(f"\nClass counts after filtering to {KEEP_CLASSES}:")
print(df_raw[LABEL_COL].value_counts())

# Drop non-numeric columns except label
numeric_cols = df_raw.select_dtypes(include=[np.number]).columns.tolist()
df = df_raw[numeric_cols + [LABEL_COL]].copy()
df = df.dropna()
print(f"\nNumeric features retained: {len(numeric_cols)}")

# Verify all classes have enough samples
for cls in KEEP_CLASSES:
    count = (df[LABEL_COL] == cls).sum()
    assert count >= TARGET_PER_CLASS, f"Class '{cls}' has only {count} samples (need {TARGET_PER_CLASS})"

# Subsample each class to TARGET_PER_CLASS
balanced_dfs = [
    df[df[LABEL_COL] == cls].sample(n=TARGET_PER_CLASS, random_state=42)
    for cls in KEEP_CLASSES
]
df = pd.concat(balanced_dfs).reset_index(drop=True)

# Per-class 80/20 stratified split
train_dfs, test_dfs = {}, {}
for cls, grp in df.groupby(LABEL_COL):
    train_part = grp.sample(frac=0.8, random_state=42)
    test_part  = grp.drop(train_part.index)
    train_dfs[cls] = train_part
    test_dfs[cls]  = test_part

df_train = pd.concat(train_dfs.values())
df_test  = pd.concat(test_dfs.values())

X_train = df_train.drop(columns=[LABEL_COL])
y_train = df_train[LABEL_COL]
X_test  = df_test.drop(columns=[LABEL_COL])
y_test  = df_test[LABEL_COL]

feature_cols = X_train.columns.tolist()

table_data = [
    [cls, TARGET_PER_CLASS, len(train_dfs[cls]), len(test_dfs[cls])]
    for cls in KEEP_CLASSES
]
print("\n" + tabulate(table_data, headers=["Class", "Total", "Train", "Test"], tablefmt="grid"))


Shape: (1194464, 49)

Class counts after filtering to ['normal', 'DoS', 'Reconn']:
Traffic
normal    1107448
DoS         78305
Reconn       8240
Name: count, dtype: int64

Numeric features retained: 44

+---------+---------+---------+--------+
| Class   |   Total |   Train |   Test |
+=========+=========+=========+========+
| normal  |    8000 |    6400 |   1600 |
+---------+---------+---------+--------+
| DoS     |    8000 |    6400 |   1600 |
+---------+---------+---------+--------+
| Reconn  |    8000 |    6400 |   1600 |
+---------+---------+---------+--------+


# ML Baseline (Multi-Class)

In [25]:
################################################################################
# Decision Tree and Random Forest baselines (multi-class)
# max_depth=5 on both models — consistent with Kerckhoffs analysis in the paper
################################################################################

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import time

os.makedirs("results/ml", exist_ok=True)

for Model, kwargs, fname in [
    (DecisionTreeClassifier, {"max_depth": 5}, "result-dt-multiclass.txt"),
    (RandomForestClassifier,  {"max_depth": 5}, "result-rf-multiclass.txt"),
]:
    model = Model(random_state=42, **kwargs)
    model.fit(X_train, y_train)
    t0      = time.time()
    y_pred  = model.predict(X_test)
    elapsed = time.time() - t0
    report  = classification_report(y_true=y_test, y_pred=y_pred, digits=4)
    matrix  = confusion_matrix(y_test, y_pred)
    print(f"=== {Model.__name__} max_depth=5 ({elapsed:.4f}s) ===")
    print(report)
    with open(f"results/ml/{fname}", "w") as f:
        f.write(f"Classification Report\n{report}\n\nConfusion Matrix\n{matrix}\n")


=== DecisionTreeClassifier max_depth=5 (0.0023s) ===
              precision    recall  f1-score   support

         DoS     1.0000    1.0000    1.0000      1600
      Reconn     1.0000    1.0000    1.0000      1600
      normal     1.0000    1.0000    1.0000      1600

    accuracy                         1.0000      4800
   macro avg     1.0000    1.0000    1.0000      4800
weighted avg     1.0000    1.0000    1.0000      4800

=== RandomForestClassifier max_depth=5 (0.0063s) ===
              precision    recall  f1-score   support

         DoS     0.9994    1.0000    0.9997      1600
      Reconn     1.0000    0.9994    0.9997      1600
      normal     0.9994    0.9994    0.9994      1600

    accuracy                         0.9996      4800
   macro avg     0.9996    0.9996    0.9996      4800
weighted avg     0.9996    0.9996    0.9996      4800



# Vector Store — Per-Class Representative Samples

In [15]:
################################################################################
# Build class_entries dict: top-10 representative rows per class via BGE-M3
################################################################################

import json
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from tqdm import tqdm

embeddings = HuggingFaceEmbeddings(
    model_name='BAAI/bge-m3',
    model_kwargs={'device': 'mps'},
    encode_kwargs={'normalize_embeddings': True, 'batch_size': 64}
)

n_results          = 10
max_embed_per_class = 100

class_entries = {}
for label in tqdm(df[LABEL_COL].unique(), desc="Building class entries"):
    class_df  = train_dfs[label].drop(columns=[LABEL_COL])
    sample_df = class_df.sample(n=min(max_embed_per_class, len(class_df)), random_state=42)
    docs      = [str(row.tolist()) for _, row in sample_df.iterrows()]
    vecs      = np.array(embeddings.embed_documents(docs))
    mean_vec  = vecs.mean(axis=0)
    norms     = np.linalg.norm(vecs, axis=1) * np.linalg.norm(mean_vec)
    sims      = (vecs @ mean_vec) / np.where(norms == 0, 1e-9, norms)
    top_idx   = np.argsort(sims)[::-1][:n_results]
    top_rows  = sample_df.iloc[top_idx]
    entries   = {col: top_rows[col].tolist() for col in feature_cols}
    class_entries[label] = entries

print("class_entries keys:", list(class_entries.keys()))

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Building class entries: 100%|██████████| 3/3 [00:16<00:00,  5.60s/it]

class_entries keys: ['normal', 'DoS', 'Reconn']


# Tool Definition

In [16]:
################################################################################
# Tool: evaluate_rule — one-vs-rest macro F1 on training set
################################################################################

from sklearn.metrics import classification_report
from tqdm import tqdm
import operator
from typing import Annotated
from langchain_core.tools import tool

show_progress = False
operations = {
    '<':  operator.lt,
    '>':  operator.gt,
    '==': operator.eq,
    '<=': operator.le,
    '>=': operator.ge,
    '!=': operator.ne,
}

@tool
def evaluate_rule(
    feature_name: Annotated[str, "Feature name"],
    value:        Annotated[str, "Value"],
    op:           Annotated[str, "Operator"],
    target_class: Annotated[str, "Target class name"]
) -> float:
    """Evaluate rule on training set. Returns macro F1 for target_class vs all others."""
    try:
        value = float(value)
    except ValueError:
        pass
    if op not in operations:
        raise ValueError(f"Unsupported operator: {op}")
    y_true, y_pred = [], []
    for lbl, df_cls in train_dfs.items():
        df_feat = df_cls.drop(columns=[LABEL_COL])
        for i in tqdm(range(len(df_feat)), disable=not show_progress,
                      desc=f"Eval {lbl[:12]}..."):
            y_true.append("target" if lbl == target_class else "other")
            y_pred.append(
                "target" if operations[op](df_feat.iloc[i][feature_name], value) else "other"
            )
    report = classification_report(y_true, y_pred, digits=4, output_dict=True, zero_division=0)
    return report['macro avg']['f1-score']

# Prompt Template

In [17]:
################################################################################
# Prompt Template
################################################################################

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

system_message = ("system",
"""You are a network security analyst specialising in IoT intrusion detection.
You are given labelled network traffic data with named features and numeric values.
Your task is to generate exactly {k} deterministic threshold rules to identify '{target_class}' traffic.

Rules must:
- Use ONLY inequality operators: '>', '<', '>=', '<='
- NEVER use '==' or '!=' — these are forbidden for numeric features
- Reference only feature names present in the data
- Be range-based thresholds that generalise beyond the exact sample values shown
- You MUST make exactly {k} tool calls — one per rule, no more, no less.

Good rule example: Header_Length < 100
Bad rule example: Header_Length == 54.0  ← FORBIDDEN"""
)

human_message = ("user",
"""Analyze the following network data and generate rules for the top {k} important features \
to identify '{target_class}' entries.

Target Class Entries:
```{target_entries}```

Other Class Entries (sample):
```{other_entries}```""")

prompt = ChatPromptTemplate.from_messages([
    system_message,
    human_message,
    MessagesPlaceholder("msgs")
])

# LLM Setup

In [18]:
################################################################################
# LLM Setup and Tool Binding
################################################################################

import dotenv
from langchain_anthropic import ChatAnthropic

dotenv.load_dotenv(os.path.join(os.getcwd(), '../.env'))

# model_name = "gpt-4o"
model_name = "claude-haiku-4-5-20251001"
# llm = ChatOpenAI(model=model_name, temperature=0.1)
llm = ChatAnthropic(model=model_name, temperature=0.1)

llm_with_tool = llm.bind_tools([evaluate_rule])

# Feedback Loop (Per Class)

In [21]:
################################################################################
# Per-class feedback loop with early stopping
# Generates and refines threshold rules for DoS and Reconn vs all others
################################################################################

import json
from langchain_core.messages import HumanMessage

chain = prompt | llm_with_tool

non_normal_classes = [cls for cls in KEEP_CLASSES if cls != NORMAL_CLASS]
print("Attack classes to generate rules for:", non_normal_classes)

CLASS_CONFIG = {
    "DoS":   {"max_rounds": 12, "k": 5},
    "Reconn": {"max_rounds": 12, "k": 5},
}

patience      = 4
show_progress = False
class_rules   = {}

os.makedirs("results/llm", exist_ok=True)
rules_output_path = "results/llm/class_rules-multiclass.json"

# Load existing checkpoint to resume if interrupted
if os.path.exists(rules_output_path):
    with open(rules_output_path) as f:
        class_rules = json.load(f)
    print(f"Resuming from checkpoint: {list(class_rules.keys())} already done")
    # Invalidate checkpoint if it was generated for a different class set
    if set(class_rules.keys()) != set(non_normal_classes):
        print(f"Checkpoint class set {set(class_rules.keys())} != current {set(non_normal_classes)} — clearing.")
        class_rules = {}

def build_other_entries(target_class, class_entries, rows_per_class=3):
    return {
        cls: list(entries)[:rows_per_class]
        for cls, entries in class_entries.items()
        if cls != target_class
    }

for target_class in non_normal_classes:
    if target_class in class_rules:
        print(f"Skipping {target_class} (already in checkpoint)")
        continue

    print(f"\n=== Target class: {target_class} ===")

    cfg        = CLASS_CONFIG[target_class]
    max_rounds = cfg["max_rounds"]
    k          = cfg["k"]

    n               = 0
    no_improve      = 0
    best_mean_f1    = 0.0
    best_tool_calls = []
    train_f1_scores = []
    msgs            = []

    target_entries = json.dumps(class_entries[target_class])
    other_entries  = json.dumps(build_other_entries(target_class, class_entries))

    while n < max_rounds and no_improve < patience:
        ai_msg = chain.invoke({
            "k": k, "target_class": target_class,
            "target_entries": target_entries,
            "other_entries": other_entries,
            "msgs": msgs
        })

        tool_msgs   = []
        rule_scores = []
        for tool_call in ai_msg.tool_calls:
            tool_msg = evaluate_rule.invoke(tool_call)
            tool_msgs.append(tool_msg)
            try:
                rule_scores.append(float(tool_msg.content))
            except (ValueError, TypeError):
                rule_scores.append(0.0)

        mean_f1 = sum(rule_scores) / len(rule_scores) if rule_scores else 0.0

        if mean_f1 > best_mean_f1:
            best_mean_f1    = mean_f1
            best_tool_calls = list(ai_msg.tool_calls)
            no_improve      = 0
        else:
            no_improve += 1

        rule_feedback_lines = []
        for i, (tc, score) in enumerate(zip(ai_msg.tool_calls, rule_scores)):
            args = tc["args"]
            rule_feedback_lines.append(
                f"  Rule {i+1}: {args['feature_name']} {args['op']} {args['value']} "
                f"-> F1={score:.4f} ({'KEEP' if score >= mean_f1 else 'REVISE'})"
            )
        rule_feedback = "\n".join(rule_feedback_lines)

        human_msg = HumanMessage(
            f"Round {n+1} results (best so far: {best_mean_f1:.4f}):\n"
            f"{rule_feedback}\n\n"
            f"Mean F1 this round: {mean_f1:.4f}. "
            f"Revise rules marked REVISE by trying different threshold values or a different feature. "
            f"Keep rules marked KEEP unchanged. "
            f"Generate exactly {k} rules for '{target_class}' and make exactly {k} tool calls."
        )

        msgs.extend([ai_msg, *tool_msgs, human_msg])
        train_f1_scores.append(mean_f1)
        n += 1

        usage_info  = ai_msg.response_metadata.get("usage", {})
        token_usage = {
            "completion_tokens": usage_info.get("output_tokens", 0),
            "prompt_tokens":     usage_info.get("input_tokens",  0),
            "total_tokens":      usage_info.get("input_tokens",  0) + usage_info.get("output_tokens", 0),
        }
        print(f"  Round: {n}  Mean F1: {mean_f1:.4f}  Best: {best_mean_f1:.4f}  "
              f"No-improve: {no_improve}  Tokens: {token_usage}")

        for tc in ai_msg.tool_calls:
            if tc["args"]["op"] in ["==", "!="]:
                print(f"  WARNING: equality operator detected — {tc['args']}")

    class_rules[target_class] = [{"args": tc["args"]} for tc in best_tool_calls]
    print(f"  Train F1 history: {train_f1_scores}")
    print(f"  Final best F1: {best_mean_f1:.4f}  (stopped after {n} rounds)")

    with open(rules_output_path, "w") as f:
        json.dump(class_rules, f, indent=2)

print("\nDone. Classes with rules:", list(class_rules.keys()))
print(f"Saved class rules to: {rules_output_path}")


Attack classes to generate rules for: ['DoS', 'Reconn']

=== Target class: DoS ===
  Round: 1  Mean F1: 0.6835  Best: 0.6835  No-improve: 0  Tokens: {'completion_tokens': 817, 'prompt_tokens': 3039, 'total_tokens': 3856}
  Round: 2  Mean F1: 0.7164  Best: 0.7164  No-improve: 0  Tokens: {'completion_tokens': 532, 'prompt_tokens': 4251, 'total_tokens': 4783}
  Round: 3  Mean F1: 0.7502  Best: 0.7502  No-improve: 0  Tokens: {'completion_tokens': 520, 'prompt_tokens': 5176, 'total_tokens': 5696}
  Round: 4  Mean F1: 0.7146  Best: 0.7502  No-improve: 1  Tokens: {'completion_tokens': 519, 'prompt_tokens': 6088, 'total_tokens': 6607}
  Round: 5  Mean F1: 0.7277  Best: 0.7502  No-improve: 2  Tokens: {'completion_tokens': 530, 'prompt_tokens': 6999, 'total_tokens': 7529}
  Round: 6  Mean F1: 0.6558  Best: 0.7502  No-improve: 3  Tokens: {'completion_tokens': 523, 'prompt_tokens': 7923, 'total_tokens': 8446}
  Round: 7  Mean F1: 0.7448  Best: 0.7502  No-improve: 4  Tokens: {'completion_tokens': 5

In [22]:
# Inspect saved rules for the first attack class
with open("results/llm/class_rules-multiclass.json") as f:
    rules = json.load(f)
first_cls = list(rules.keys())[0]
print(f"Rules for '{first_cls}':")
print(json.dumps(rules[first_cls], indent=2))

Rules for 'DoS':
[
  {
    "args": {
      "feature_name": "DstBytes",
      "op": "<=",
      "value": "0",
      "target_class": "DoS"
    }
  },
  {
    "args": {
      "feature_name": "DstLoad",
      "op": "<=",
      "value": "0",
      "target_class": "DoS"
    }
  },
  {
    "args": {
      "feature_name": "pLoss",
      "op": ">=",
      "value": "33",
      "target_class": "DoS"
    }
  },
  {
    "args": {
      "feature_name": "DstPkts",
      "op": "<=",
      "value": "0",
      "target_class": "DoS"
    }
  },
  {
    "args": {
      "feature_name": "DstRate",
      "op": "<=",
      "value": "0",
      "target_class": "DoS"
    }
  }
]


In [23]:
################################################################################
# Evaluate generated rules on test set (multi-class)
################################################################################

import json
import operator
from sklearn.metrics import classification_report, confusion_matrix
from tqdm import tqdm

operations = {
    '<':  operator.lt,
    '>':  operator.gt,
    '==': operator.eq,
    '<=': operator.le,
    '>=': operator.ge,
    '!=': operator.ne,
}

rules_input_path = "results/llm/class_rules-multiclass.json"
with open(rules_input_path, "r") as f:
    class_rules = json.load(f)

print(f"Loaded {len(class_rules)} attack classes")
print("Rule counts:", {c: len(r) for c, r in class_rules.items()})

def predict_multiclass(row, class_rules):
    scores = {}
    for cls, tool_calls in class_rules.items():
        if not tool_calls:
            scores[cls] = 0.0
            continue
        count = 0
        for tc in tool_calls:
            args = tc["args"]
            op, feat, val = args["op"], args["feature_name"], args["value"]
            try:
                val = float(val)
            except ValueError:
                pass
            if op in operations and feat in row.index:
                if operations[op](row[feat], val):
                    count += 1
        scores[cls] = count / len(tool_calls)

    max_score = max(scores.values()) if scores else 0.0
    if max_score == 0.0:
        return NORMAL_CLASS

    winners = [cls for cls, s in scores.items() if s == max_score]
    return winners[0] if len(winners) == 1 else NORMAL_CLASS

y_pred_llm, y_true_llm = [], []
for i in tqdm(range(len(X_test)), desc="Evaluating multiclass rules"):
    row = X_test.iloc[i]
    y_true_llm.append(y_test.iloc[i])
    y_pred_llm.append(predict_multiclass(row, class_rules))

report = classification_report(y_true_llm, y_pred_llm, digits=4)
matrix = confusion_matrix(y_true_llm, y_pred_llm)
print(report)

os.makedirs("results/llm", exist_ok=True)
with open("results/llm/result-llm-multiclass.txt", "w") as f:
    f.write(f"Classification Report\n{report}\n\nConfusion Matrix\n{matrix}\n")

Loaded 2 attack classes
Rule counts: {'DoS': 5, 'Reconn': 5}


Evaluating multiclass rules: 100%|██████████| 4800/4800 [00:00<00:00, 29610.99it/s]

              precision    recall  f1-score   support

         DoS     0.6927    0.6863    0.6895      1600
      Reconn     0.9944    0.3325    0.4984      1600
      normal     0.5840    0.9781    0.7313      1600

    accuracy                         0.6656      4800
   macro avg     0.7570    0.6656    0.6397      4800
weighted avg     0.7570    0.6656    0.6397      4800



# Efficiency Comparison

In [24]:
################################################################################
# Inference latency: DT vs RF vs LLM policy rules
################################################################################

import time
from tabulate import tabulate

model_dt = DecisionTreeClassifier(random_state=42)
model_rf = RandomForestClassifier(random_state=42)
model_dt.fit(X_train, y_train)
model_rf.fit(X_train, y_train)

elapsed_dt, elapsed_rf, elapsed_llm = [], [], []
n_samples = min(1000, len(X_test))

for i in range(n_samples):
    row = X_test.iloc[[i]]

    t0 = time.time()
    model_dt.predict(row)
    elapsed_dt.append(time.time() - t0)

    t0 = time.time()
    model_rf.predict(row)
    elapsed_rf.append(time.time() - t0)

    t0 = time.time()
    predict_multiclass(X_test.iloc[i], class_rules)
    elapsed_llm.append(time.time() - t0)

rows = [
    ["Decision Tree", f"{sum(elapsed_dt)/n_samples*1e3:.4f} ms"],
    ["Random Forest", f"{sum(elapsed_rf)/n_samples*1e3:.4f} ms"],
    ["LLM Rules",     f"{sum(elapsed_llm)/n_samples*1e3:.4f} ms"],
]
print(tabulate(rows, headers=["Model", "Avg Inference Time"], tablefmt="grid"))

+---------------+----------------------+
| Model         | Avg Inference Time   |
+===============+======================+
| Decision Tree | 0.3682 ms            |
+---------------+----------------------+
| Random Forest | 2.0128 ms            |
+---------------+----------------------+
| LLM Rules     | 0.0442 ms            |
+---------------+----------------------+
